# Image Classification

Define function for reading images. Files names must begin with species name.

In [ ]:
import os
import numpy as np
from PIL import Image

# Define the image size
img_width, img_height = 400, 300

# Function to load and preprocess images
def load_images(path):
    images = []
    labels = []
    for img_file in os.listdir(path):
        img_path = os.path.join(path, img_file)
        try:
            img = Image.open(img_path).convert('RGB')
            img = img.resize((img_width, img_height))
            img_array = np.array(img)
            images.append(img_array)
            species = img_file.split('_', 1)[0]
            labels.append(species)
        except (OSError, FileNotFoundError):
            print(f"Error loading image: {img_path}. Skipping.")
    return np.array(images), np.array(labels)

## Read all files

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Define the path to your dataset
dataset_path = "training-images/"

# Load the images and labels
images, labels = load_images(dataset_path)

# Convert labels to numerical values using LabelEncoder
label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)

# Display the unique classes
print("Classes:", label_encoder.classes_)

# Prepare Training Data

Convert images to flat arrays and split off a test data set.

In [ ]:
# Flatten the images (convert 3D array to 1D array)
images_flattened = images.reshape(images.shape[0], -1)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(images_flattened, labels_encoded, test_size=0.1, random_state=23)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

## Train the Model

Fit a classifier to the images, test with the test data set.

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score

# Create an Support Vector model
model = SVC(kernel='rbf', random_state=42)

# Train the model
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate some metrics
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')

## Visualize the Problem

See which animals are classified wrong to improve your training data.

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Create a confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Visualize the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()


## Export to File

In [ ]:
import pickle

# save the model to file
with open('animals.pkl', 'wb') as handle:
    pickle.dump(model, handle, protocol=pickle.HIGHEST_PROTOCOL)

# save the label names
with open('animals-labels.pkl', 'wb') as handle:
    pickle.dump(label_encoder, handle, protocol=pickle.HIGHEST_PROTOCOL)